# Arxiv-Scholar: End-to-End RAG Pipeline

This notebook walks through the entire ingestion and retrieval pipeline of the **Arxiv-Scholar** project. We will explore each component in detail, highlighting their advantages and internal workings.

The pipeline consists of the following phases:
1. **Download & Ingestion:** Fetching scientific papers from arXiv and parsing them.
2. **Chunking:** Splitting the parsed documents into layout-aware semantic chunks.
3. **Embedding:** Generating both Dense (Semantic) and Sparse (Keyword) vectors using lightweight CPU models.
4. **Vector Database Insertion:** Storing chunks and vectors in Qdrant.
5. **Hybrid Retrieval:** Querying Qdrant using Reciprocal Rank Fusion (RRF) to combine Dense and Sparse search results.

In [1]:
import os
import sys
import logging
import json
import pprint

sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Ensure we have a download directory
os.environ["DOWNLOAD_DIR"] = "notebook_trial"
os.environ["STATE_FILE"] = "notebook_trial_state.json"

## 1. Download & Ingestion

**Components:** `ArxivUnifiedEngine` and `LocalDirectoryReader`

**Concept:** Ingesting 1TB of arXiv PDFs requires careful batching to avoid running out of disk space and memory. The `ArxivUnifiedEngine` downloads PDFs in manageable batches based on state tracking. The `LocalDirectoryReader` then parses these unstructured PDF files into a structured `Document` schema.

**Advantages:**
- Resume-able ingestion: the state tracking ensures we can pause/resume.
- Memory efficient: files are processed locally in small batches and then cleaned up.

In [2]:
from arxiv_scholar.download.arxiv_ingestion import ArxivUnifiedEngine
from arxiv_scholar.ingestion.local import LocalDirectoryReader
from configs import config

config.DOWNLOAD_DIR = "notebook_trial"
config.STATE_FILE = "notebook_trial_state.json"

engine = ArxivUnifiedEngine()
print("Fetching a small batch of PDFs (batch_size=1)...")
paths = engine.get_batch(batch_size=1)

print(f"\nDownloaded paths: {paths}")

print("\nReading documents from local directory...")
reader = LocalDirectoryReader(directory_path=config.DOWNLOAD_DIR)
documents = list(reader.read())

print(f"\nExtracted {len(documents)} document(s).")
if documents:
    sample_doc = documents[0]
    print(f"Document ID: {sample_doc.id}")
    print(f"Document Metadata: {sample_doc.metadata}")
    print(f"Document Content (First 200 chars): {sample_doc.content[:200]}...")

/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Discovering historical arXiv folders...
Fetching a small batch of PDFs (batch_size=1)...

Downloaded paths: ['notebook_trial/0704.0003v1.pdf']

Reading documents from local directory...
MuPDF error: format error: No default Layer config


Extracted 3 document(s).
Document ID: 79a83739fa5af0d84fc56eef52df48d7e41f278f4e7006b1dfb17921c121cf47
Document Metadata: {'source_path': '/Users/tri/Projects/arxiv-scholar/notebooks/notebook_trial/0704.0003v1.pdf', 'filename': '0704.0003v1.pdf', 'file_size_bytes': 132664, 'title': 'The evolution of the Earth-Moon system based on the dark fluid model', 'arxiv_id': '0704.0003v1'}
Document Content (First 200 chars): The evolution of the Earth-Moon system based on  
the dark matter field fluid model 
 
Hongjun, Pan 
Department of Chemistry, University of North Texas 
Denton, TX 76203, U. S. A 
 
Abstract 
The evol...


## 2. Layout-Aware Chunking

**Component:** `LayoutAwareChunker`

**Concept:** Scientific papers are dense with information. Simply splitting by character count slices paragraphs and tables in half, destroying semantic meaning. Layout-aware chunking attempts to split text at natural boundaries (paragraphs, sections) up to a defined token limit.

**Advantages:**
- Preserves context boundaries.
- Embeddings are more accurate because the chunk contains a complete thought.

In [8]:
from arxiv_scholar.chunking.layout import LayoutAwareChunker

chunker = LayoutAwareChunker(max_chunk_size=1500)
chunks = []

if documents:
    print(f"Chunking document {sample_doc.id}...")
    chunks = list(chunker.chunk(sample_doc))
    print(f"Generated {len(chunks)} chunks.")
    if chunks:
        print(f"\nSample Chunk [0] ID: {chunks[0].id}")
        print(f"Sample Chunk [0] Length: {len(chunks[0].content)} chars")
        print(f"Sample Chunk [0] Preview: {chunks[0].content[:200]}...")

for chunk in chunks:
    print(f"\nChunk ID: {chunk.id}")
    print(f"Preview {chunk.content[:200]}...")

2026-05-26 11:40:53,999 [ERROR] docling is not installed. Please install it with `pip install docling` to use the LayoutAwareChunker.
2026-05-26 11:40:54,000 [WARNING] Docling not available. Falling back to sliding window chunking.


Chunking document 79a83739fa5af0d84fc56eef52df48d7e41f278f4e7006b1dfb17921c121cf47...
Generated 22 chunks.

Sample Chunk [0] ID: 112aa5119a1fbbe61ca2cc65d87d45f96b294a3532cbbb9104480c69e7281e98
Sample Chunk [0] Length: 1498 chars
Sample Chunk [0] Preview: The evolution of the Earth-Moon system based on  
the dark matter field fluid model 
 
Hongjun, Pan 
Department of Chemistry, University of North Texas 
Denton, TX 76203, U. S. A 
 
Abstract 
The evol...

Chunk ID: 112aa5119a1fbbe61ca2cc65d87d45f96b294a3532cbbb9104480c69e7281e98
Preview The evolution of the Earth-Moon system based on  
the dark matter field fluid model 
 
Hongjun, Pan 
Department of Chemistry, University of North Texas 
Denton, TX 76203, U. S. A 
 
Abstract 
The evol...

Chunk ID: dc088cb19275a486c746788cd56656938e7c7ddf3eebb485e31456ef62580e0c
Preview (Hartmann and Davis 1975). Since the 
formation of the Earth-Moon system, it has been evolving at all time scale. It is well 
known that the Moon is receding from us an

## 3. Embedding (Dense + Sparse)

**Components:** `FastEmbedEmbedder` and `SparseBM25Embedder`

**Concept:** 
- **Dense Embeddings:** Capture the semantic meaning of text in a dense vector (e.g., 384 dimensions). Useful for "conceptual" searches where the exact words don't match but the meaning does.
- **Sparse Embeddings (BM25):** Capture exact keyword frequencies. Useful for searching specific names, acronyms, or formulas.

We use `fastembed` which uses ONNX Runtime. 
**Advantages:**
- No heavy PyTorch dependency, much smaller footprint.
- Highly optimized for CPU inference.

In [4]:
from arxiv_scholar.embedding.fastembed_embedder import FastEmbedEmbedder, SparseBM25Embedder

print("Loading FastEmbed Dense Model (BAAI/bge-small-en-v1.5)...")
dense_embedder = FastEmbedEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    batch_size=2
)

print("Loading FastEmbed Sparse Model (Qdrant/bm25)...")
sparse_embedder = SparseBM25Embedder(batch_size=2)

if chunks:
    # Take top 5 chunks for speed in this notebook
    sample_chunks = chunks[:5]
    texts = [c.content for c in sample_chunks]
    
    print(f"\nEmbedding {len(texts)} chunks (Dense)...")
    dense_vectors = dense_embedder.embed(texts)
    print(f"Dense Vector [0] length: {len(dense_vectors[0])} dimensions")
    
    print(f"\nEmbedding {len(texts)} chunks (Sparse)...")
    sparse_vectors = sparse_embedder.embed(texts)
    print(f"Sparse Vector [0] Indices shape: {len(sparse_vectors[0].indices)}")
    print(f"Sparse Vector [0] Values shape: {len(sparse_vectors[0].values)}")

Loading FastEmbed Dense Model (BAAI/bge-small-en-v1.5)...


2026-05-26 11:32:28,556 [INFO] Loading FastEmbed model 'BAAI/bge-small-en-v1.5' (CPU/ONNX)...
2026-05-26 11:32:28,714 [INFO] Model loaded. Dimension: 384, Device: CPU (ONNX)
2026-05-26 11:32:28,714 [INFO] Loading FastEmbed sparse model 'Qdrant/bm25'...


Loading FastEmbed Sparse Model (Qdrant/bm25)...

Embedding 5 chunks (Dense)...
Dense Vector [0] length: 384 dimensions

Embedding 5 chunks (Sparse)...
Sparse Vector [0] Indices shape: 104
Sparse Vector [0] Values shape: 104


## 4. Vector Database Insertion

**Component:** `QdrantVectorStore`

**Concept:** We need a robust database to store and index our high-dimensional vectors and sparse vectors. Qdrant is chosen for its native support of multi-vector search (hybrid search) and its high performance.

**Advantages:**
- Supports `Prefetch` to run sparse and dense searches simultaneously.
- Native Reciprocal Rank Fusion (RRF) on the server side, saving network bandwidth.

In [5]:
from arxiv_scholar.storage.qdrant_store import QdrantVectorStore

collection_name = "arxiv_notebook_test"

print("Connecting to Qdrant (ensure server is running on localhost:6333)...")
store = QdrantVectorStore(
    collection_name=collection_name,
    host="localhost",
    port=6333
)

print("Ensuring collection exists with correct dimensions...")
store.ensure_collection(dimension=dense_embedder.dimension)

if chunks:
    print(f"\nUpserting {len(sample_chunks)} points into Qdrant...")
    upserted_count = store.upsert(
        chunks=sample_chunks,
        vectors=dense_vectors,
        sparse_vectors=sparse_vectors
    )
    print(f"Successfully upserted {upserted_count} points.")

2026-05-26 11:34:12,447 [INFO] QdrantVectorStore connected to localhost:6333, collection=arxiv_notebook_test
2026-05-26 11:34:12,450 [INFO] HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
2026-05-26 11:34:12,451 [INFO] Collection 'arxiv_notebook_test' already exists.
2026-05-26 11:34:12,455 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-26 11:34:12,457 [INFO] HTTP Request: PUT http://localhost:6333/collections/arxiv_notebook_test/points?wait=true "HTTP/1.1 200 OK"


Connecting to Qdrant (ensure server is running on localhost:6333)...
Ensuring collection exists with correct dimensions...

Upserting 5 points into Qdrant...
Successfully upserted 5 points.


/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## 5. Hybrid Retrieval using RRF

**Component:** `HybridRetriever`

**Concept:** When a user asks a question, we want the best of both worlds: conceptual matching (Dense) and exact term matching (Sparse). We issue both queries to Qdrant using the `Prefetch` API. Qdrant runs both searches independently, and then fuses the ranked lists together using Reciprocal Rank Fusion (RRF).

**Formula (RRF):** $Score = \frac{1}{k + Rank_{dense}} + \frac{1}{k + Rank_{sparse}}$

In [13]:
from arxiv_scholar.retrieval.retrieval import HybridRetriever

print("Initializing Hybrid Retriever...")
retriever = HybridRetriever(
    collection_name=collection_name,
    qdrant_host="localhost",
    qdrant_port=6333,
    dense_model_name="BAAI/bge-small-en-v1.5",  # Match model used in ingestion
    sparse_model_name="Qdrant/bm25"
)

query = "What is dark matter field fluid?"
print(f"\nExecuting Hybrid Search for: '{query}'")
results = retriever.retrieve(query_text=query, limit=3)

print("\n--- Top Results ---")
for i, res in enumerate(results):
    print(f"\nRank {i+1} | Score: {res['score']:.4f}")
    print(f"Chunk ID: {res['chunk_id']}")
    print(f"Content preview: {res['text'][:250]}...")

2026-05-26 11:48:43,209 [INFO] Initialized QdrantClient for collection 'arxiv_notebook_test'
2026-05-26 11:48:43,209 [INFO] Loading dense model: BAAI/bge-small-en-v1.5
2026-05-26 11:48:43,214 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Initializing Hybrid Retriever...


2026-05-26 11:48:43,416 [INFO] Loading sparse model: Qdrant/bm25



Executing Hybrid Search for: 'What is dark matter field fluid?'


Traceback (most recent call last):
  File "/Users/tri/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev/_pydevd_bundle/pydevd_comm.py", line 735, in make_thread_stack_str
    append('file="%s" line="%s">' % (make_valid_xml_value(my_file), lineno))
  File "/Users/tri/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev/_pydevd_bundle/pydevd_xml.py", line 36, in make_valid_xml_value
    return s.replace("&", "&amp;").replace('<', '&lt;').replace('>', '&gt;').replace('"', '&quot;')
AttributeError: 'tuple' object has no attribute 'replace'


KeyboardInterrupt: 

## Cleanup
Clean up the downloaded batch to free disk space.

In [10]:
print("Cleaning up batch...")
engine.cleanup_batch(paths)
print("Done!")

Cleaning up batch...
Done!
